# Cross-technology alignment: Xenium -> Visium (dlPFC, Br8667)

**STalign LDDMM pipeline** registering single-cell Xenium onto spot-level Visium for one donor
of the Lieber schizophrenia dlPFC cohort (Kwon/Guo 2026, GSE307404).

Each numbered section is a self-contained cell; run in order after a kernel restart.
Cells print a `PASS/CHECK` self-check so failures are caught early.

**Pipeline:** load + QC -> attach official Lieber annotations -> layer axis -> initial affine
-> rasterize -> LDDMM (both mirrors) -> apply + non-circular layer validation -> concordance -> persist.

**Key result (see Methods at the end):** the alignment robustly recovers the GM/WM axis
(+/-1 layer ~74%, 63 sigma over random) but not intra-GM laminar correspondence -- a geometric
limit of tangentially homogeneous cortex, not a tunable parameter.

## 1 - Environment, physical constants, sample paths

In [ ]:
# ============================================================
#  Cell 1 - environment + physical constants (Br8667)
#  Physical scale is fixed from hardware, NOT fitted:
#    - Xenium cell coordinates are ALREADY in microns
#    - Visium coordinates are full-res pixels; the only reliable ruler
#      is the fixed 100 um spot pitch, so um/px is back-computed from it
#      (never the section-specific 0.73 in some scalefactors_json).
# ============================================================
import os, glob, re, json, warnings, gzip, io
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, scanpy as sc, anndata as ad
from scipy.io import mmread
from scipy.spatial import cKDTree
from scipy.stats import spearmanr, mannwhitneyu, rankdata
from STalign import STalign

device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch.set_default_device(device)
for msg in (".*match groups.*", ".*To copy construct.*",
            ".*no positive values.*", ".*meshgrid.*", ".*tight_layout.*"):
    warnings.filterwarnings("ignore", message=msg)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 600, "axes.grid": False, "font.size": 10})

# ---- fixed physical constants ----
SPOT_DIAM  = 55.0     # Visium spot diameter (um)
SPOT_PITCH = 100.0    # Visium spot centre-to-centre (um) -> the scale anchor
DX_SOURCE, DX_TARGET = 30.0, 100.0   # rasterization grid: um (Xenium) / px (Visium)
GRID_UM      = 200.0  # pixel size for expression correspondence (paper value)
MATCH_THRESH = 0.85   # STalign matching-probability cutoff for "matched" cells
DTYPE        = torch.float64

SPD_ORDER  = ["SpD07-L1","SpD06-L2/3","SpD02-L3/4","SpD05-L5","SpD03-L6","SpD04-WM","SpD01-WMtz"]
MARK_UPPER = ["CUX2","LAMP5","RORB"]
MARK_WM    = ["MBP","PLP1","MOBP","MOG","CLDN11"]

SAMPLE = "Br8667"
XDIR   = "/home/yf23c/Guo_xenium/xenium"
VDIR   = f"/home/yf23c/Guo_xenium/{SAMPLE}_export"
OUT    = f"/home/yf23c/Guo_xenium/stalign_out/{SAMPLE}"; os.makedirs(OUT, exist_ok=True)

# Xenium cells file may be .parquet OR .parquet.gz depending on the donor
hits = (glob.glob(f"{XDIR}/GSM*_{SAMPLE}-cells.parquet") +
        glob.glob(f"{XDIR}/GSM*_{SAMPLE}-cells.parquet.gz"))
assert len(hits) >= 1, f"no Xenium cells file for {SAMPLE}"
CELLS_PATH = hits[0]
XSTEM = CELLS_PATH.replace("-cells.parquet.gz","").replace("-cells.parquet","")
print(f"sample {SAMPLE} | device {device} | Xenium stem {os.path.basename(XSTEM)}")


## 2 - Load Xenium + Visium, QC, physical scale

Xenium cells are **flagged, not dropped** -- LDDMM needs the full cell density.
The 1-based row index in the cells file is stored as `orig_row`; this is the key used
to join Lieber's per-cell annotations in the next cell.

In [ ]:
# ============================================================
#  Cell 2 - load, QC (flag outliers), compute physical scale
# ============================================================
def read_parquet_any(path):
    # .parquet.gz = parquet wrapped in gzip; pyarrow cannot read the outer
    # gzip, so decompress in memory first.
    with open(path, "rb") as f:
        is_gz = f.read(2) == b"\x1f\x8b"
    if is_gz:
        with gzip.open(path, "rb") as f:
            return pd.read_parquet(io.BytesIO(f.read()))
    return pd.read_parquet(path)

# ---- Xenium: 300-gene panel + centroids ----
cells = read_parquet_any(CELLS_PATH).reset_index(drop=True)
cells["orig_row"] = np.arange(len(cells)) + 1        # 1-based, = Lieber key suffix
adata = sc.read_10x_h5(f"{XSTEM}-cell_feature_matrix.h5"); adata.var_names_make_unique()
adata = adata[cells["cell_id"].astype(str).values].copy()
adata.obsm["spatial"]   = cells[["x_centroid","y_centroid"]].to_numpy(float)
adata.obs["orig_row"]   = cells["orig_row"].values
adata.obs["cell_id_xe"] = cells["cell_id"].astype(str).values
for c in ("cell_area","nucleus_area"):
    if c in cells: adata.obs[c] = cells[c].to_numpy()

is_ctrl = adata.var_names.str.match(
    r"^(NegControlProbe|NegControlCodeword|Unassigned|BLANK|Deprecated|Intergenic)", case=False)
X = adata.X.toarray() if hasattr(adata.X,"toarray") else np.asarray(adata.X)
adata = adata[X.sum(1) > 0].copy(); X = X[X.sum(1) > 0]
def mad_out(v, t="both", n=3):                       # Lieber-style MAD outlier flag
    v = np.asarray(v,float); m = np.median(v); s = np.median(np.abs(v-m))*1.4826
    return {"lower": v < m-n*s, "both": (v < m-n*s)|(v > m+n*s)}[t]
adata.obs["total_counts"] = X.sum(1)
adata.obs["detected"]     = (X > 0).sum(1)
adata.obs["is_outlier"]   = mad_out(X.sum(1),"both") | mad_out((X>0).sum(1),"lower")
adata = adata[:, ~is_ctrl].copy()
xI, yI = adata.obsm["spatial"].T

# ---- Visium: flat SPE export (barcode as a column, ENSG -> symbol) ----
rl = lambda p: [l.strip().strip('"') for l in open(p) if l.strip()]
bc, gn = rl(f"{VDIR}/{SAMPLE}_barcodes.tsv"), rl(f"{VDIR}/{SAMPLE}_genes.tsv")
orient = lambda M: (M.T if M.shape == (len(gn), len(bc)) else M).tocsr()
vadata = ad.AnnData(X=orient(mmread(f"{VDIR}/{SAMPLE}_counts.mtx").tocsr()))
if os.path.exists(f"{VDIR}/{SAMPLE}_logcounts.mtx"):
    vadata.layers["logcounts"] = orient(mmread(f"{VDIR}/{SAMPLE}_logcounts.mtx").tocsr())
vadata.obs_names, vadata.var_names = bc, gn
vadata.obs = vadata.obs.join(pd.read_csv(f"{VDIR}/{SAMPLE}_metadata.csv").set_index("barcode"))
coord = pd.read_csv(f"{VDIR}/{SAMPLE}_coordinates.csv").set_index("barcode")
vadata.obsm["spatial"] = coord.loc[bc, ["x","y"]].to_numpy(float)   # full-res pixels
gann = pd.read_csv(f"{VDIR}/{SAMPLE}_gene_annotation.csv").drop_duplicates("gene_id").set_index("gene_id")
mapped = vadata.var_names.map(gann["gene_name"])
vadata.var["gene_id"] = vadata.var_names.values
vadata.var_names = np.where(mapped.notna(), mapped, vadata.var_names)
vadata.var_names_make_unique()
xJ, yJ = vadata.obsm["spatial"].T

# ---- physical scale: Visium um/px from the fixed 100 um pitch ----
pitch_px  = np.median(cKDTree(np.c_[xJ,yJ]).query(np.c_[xJ,yJ], k=2)[0][:,1])
PX_PER_UM = pitch_px / SPOT_PITCH
UM_PER_PX = SPOT_PITCH / pitch_px
SPOT_R_PX = pitch_px * SPOT_DIAM / SPOT_PITCH / 2
shared = sorted(set(adata.var_names) & set(vadata.var_names))
xJ_um, yJ_um = xJ*UM_PER_PX, yJ*UM_PER_PX
dx = vadata.obs["DX"].unique()[0] if "DX" in vadata.obs else "?"

print(f"Xenium : {adata.n_obs:>6} cells x {adata.n_vars} genes | "
      f"{xI.ptp()/1000:.1f} x {yI.ptp()/1000:.1f} mm | flagged {adata.obs.is_outlier.mean()*100:.1f}%")
print(f"Visium : {vadata.n_obs:>6} spots x {vadata.n_vars} genes | "
      f"{xJ_um.ptp()/1000:.1f} x {yJ_um.ptp()/1000:.1f} mm | dx {dx}")
print(f"scale  : {UM_PER_PX:.4f} um/px | spot R {SPOT_R_PX:.1f} px | shared {len(shared)} genes")
chk = {"shared==300": len(shared)==300, "Xenium wider (um)": xI.ptp()>xJ_um.ptp(),
       "um/px in 0.3-1.0": 0.3<UM_PER_PX<1.0, "flagged<15%": adata.obs.is_outlier.mean()<0.15,
       "orig_row present": "orig_row" in adata.obs}
print("--- self-check ---"); [print(f"  [{'PASS' if v else 'CHECK'}] {k}") for k,v in chk.items()]
print("ALL PASS" if all(chk.values()) else ">>> some CHECK <<<")


## 3 - Attach official Lieber annotations (layer + cell type)

Lieber committed two per-cell CSVs (GitHub issue #1). The key is `{BrNum}_{orig_row}`,
matching how `01_build_spe.R` named columns. These labels are **independent of the Visium
alignment** (BANKSY / label-transfer on Xenium itself), so downstream layer-concordance is
non-circular. Missing keys = Lieber's QC outliers. The join is *verified*: the official layer
must agree spatially with the marker-derived layer axis (Spearman rho ~ 0.86).

In [ ]:
# ============================================================
#  Cell 3 - join Lieber layer + cell-type labels, verify the join
# ============================================================
BASE = ("https://raw.githubusercontent.com/LieberInstitute/"
        "spatialDLPFC_SCZ_XENIUM/devel/processed-data")
spd = pd.read_csv(f"{BASE}/04_label_transfer/label_transfer_N24_k50_smoothed_labels.csv")
spd.columns = ["key","predictions_smooth"]; spd = spd.set_index("key")["predictions_smooth"]
ct  = pd.read_csv(f"{BASE}/06_cell_type_clustering/banksy_clustering_lambda0.1_res0.7.csv")
ct  = ct.set_index("V2")["V1"]

DOMAIN2LAYER = {"spd07":"L1/M","spd06":"L2/3","spd02":"L3/4","spd05":"L5",
                "spd03":"L6","spd04":"WM","spd01":"WMtz"}
CLUSTER2CT   = {2:"L2/3 Ex",11:"L4/5 Ex",18:"L5 Ex",14:"L6 Ex",9:"In: SST,PVALB",
                12:"In: VIP,LAMP5",6:"Oligo",8:"Oligo",1:"Oligo",10:"Ambig/Oligo",
                7:"Mic",17:"Ast",16:"Ast",4:"Ast",5:"Endo",13:"Endo",3:"Endo",15:"Ambig/In/Endo"}

keys = pd.Series([f"{SAMPLE}_{r}" for r in adata.obs["orig_row"]], index=adata.obs_names)
adata.obs["lieber_key"]   = keys.values
adata.obs["spd"]          = keys.map(spd).values
adata.obs["layer"]        = adata.obs["spd"].map(DOMAIN2LAYER)
adata.obs["banksy_clust"] = keys.map(ct).values
adata.obs["cell_type"]    = adata.obs["banksy_clust"].map(CLUSTER2CT)

# marker layer axis (independent of Lieber) for cross-validation:
# library-normalise BEFORE z-scoring so total-count does not confound the score
def marker_score(A, genes, layer=None):
    g = [x for x in genes if x in A.var_names]
    M = A[:, g].layers[layer] if layer else A[:, g].X
    M = M.toarray() if hasattr(M,"toarray") else np.asarray(M)
    if layer is None:
        lib = np.asarray(A.X.sum(1)).ravel()[:,None].astype(float); lib[lib==0]=1
        M = np.log1p(M/lib*1e4)
    return ((M-M.mean(0))/(M.std(0)+1e-9)).mean(1)
marker_axis = marker_score(adata, MARK_WM) - marker_score(adata, MARK_UPPER)

LAY = ["L1/M","L2/3","L3/4","L5","L6","WM","WMtz"]
lay_code = adata.obs["layer"].map({l:i for i,l in enumerate(LAY)}).astype(float).to_numpy()
okm = ~np.isnan(lay_code)
prof = pd.DataFrame({"lay":lay_code[okm],"ax":marker_axis[okm]}).groupby("lay")["ax"].mean()
rho = spearmanr(prof.index, prof.values).statistic
cov = adata.obs["layer"].notna().mean()
print(f"annotation coverage {cov*100:.1f}% (missing = Lieber QC outliers)")
print(f"[{'PASS' if cov>0.8 else 'CHECK'}] coverage")
print(f"[{'PASS' if rho>0.8 else 'CHECK'}] join correct: Spearman(Lieber layer, marker axis) = {rho:.3f}")
print("\ncell-type counts:\n", adata.obs["cell_type"].value_counts().to_string())


## 4 - Layer axis for rasterization

The signal that drives alignment. Primary source is the **Lieber official layer ordinal
(1 = pia ... 7 = WM)**; the ~4% of cells that are Lieber QC outliers are filled by marker
rank so the axis has no NaNs (rasterization cannot take NaN). Visium uses its `spd_label` ordinal.

In [ ]:
# ============================================================
#  Cell 4 - build the 1..7 layer axis (Xenium) and spd ordinal (Visium)
# ============================================================
lay_ord = adata.obs["layer"].map({l:i+1 for i,l in enumerate(LAY)})
has_off = lay_ord.notna().values
la = np.full(adata.n_obs, np.nan)
la[has_off] = lay_ord.values[has_off].astype(float)
if (~has_off).any():                                  # fill outliers with marker rank on 1..7
    mk = marker_axis[~has_off]
    la[~has_off] = 1 + 6*(rankdata(mk)-1)/(len(mk)-1)
layer_axis = la                                       # 1 (pia) .. 7 (WM)
lab = vadata.obs["spd_label"].map({d:i+1 for i,d in enumerate(SPD_ORDER)}).to_numpy(float)

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for a, coord, s, ttl in [(ax[0], xI, layer_axis, "Xenium layer axis vs x"),
                         (ax[1], yJ, lab, "Visium spd vs y")]:
    b = pd.cut(coord, 25)
    m = pd.DataFrame({"c":coord,"s":s,"b":b}).groupby("b",observed=True).mean()
    a.plot(m["c"], m["s"], "-o", ms=3, c="#0072B2"); a.set_title(ttl)
plt.tight_layout(); plt.show()
chk = {"layer_axis 1..7": np.nanmin(layer_axis)>=1 and np.nanmax(layer_axis)<=7,
       "no NaN": not np.isnan(layer_axis).any(), "spd_label valid": not np.isnan(lab).any()}
print("--- self-check ---"); [print(f"  [{'PASS' if v else 'CHECK'}] {k}") for k,v in chk.items()]
print("ALL PASS" if all(chk.values()) else ">>> some CHECK <<<")


## 5 - Initial affine (fixed 90 deg, mirror decided later)

Scale is the physical `PX_PER_UM`, not a bounding-box fit. Rotation is fixed at 90 deg
(section-specific; set by inspecting the layer topology of the two sections). Both mirror
states are built and carried forward -- outline IoU is degenerate for this tissue (~0.42 vs
0.42), so the mirror is decided by LDDMM matched fraction in the next-but-one cell.

In [ ]:
# ============================================================
#  Cell 5 - initial affine at 90 deg, both mirrors
# ============================================================
th = np.radians(90)
Rrot = np.array([[np.cos(th),-np.sin(th)],[np.sin(th),np.cos(th)]])
def build_A(mirror):
    M = Rrot @ np.diag([1,-1]) if mirror else Rrot
    S = PX_PER_UM * M
    src, dst = np.c_[xI,yI], np.c_[xJ,yJ]
    t = dst.mean(0) - (src @ S.T).mean(0)             # centroid alignment
    A = np.eye(3)                                     # STalign acts on (y, x, 1)
    A[0,0],A[0,1],A[0,2] = S[1,1],S[1,0],t[1]
    A[1,0],A[1,1],A[1,2] = S[0,1],S[0,0],t[0]
    return A, src @ S.T + t
def outline_iou(pts, ref, bs=150.0):
    lo, hi = np.minimum(pts.min(0),ref.min(0)), np.maximum(pts.max(0),ref.max(0))
    bn = [np.arange(lo[i],hi[i]+bs,bs) for i in (0,1)]
    h1,h2 = np.histogram2d(*pts.T,bins=bn)[0]>0, np.histogram2d(*ref.T,bins=bn)[0]>0
    return (h1&h2).sum()/(h1|h2).sum()
ref = np.c_[xJ,yJ]
cands = {m: build_A(m) for m in (False, True)}
cands = {m: (A, out, outline_iou(out, ref)) for m,(A,out) in cands.items()}
for m,(A,_,iou) in cands.items():
    print(f"mirror={str(m):5s} det={np.linalg.det(A[:2,:2]):+.3f} IoU={iou:.3f}")
print("IoU degenerate - LDDMM decides mirror in the next cell")

fig, ax = plt.subplots(1, 3, figsize=(19, 6))
for a, m in zip(ax[:2], (False, True)):
    _, out, iou = cands[m]
    a.scatter(xJ, yJ, s=8, c=lab, cmap="viridis", alpha=.5, linewidths=0)
    a.scatter(*out.T, s=1, c=layer_axis, cmap="autumn", vmin=1, vmax=7, alpha=.35, linewidths=0)
    a.set_title(f"mirror={m} IoU={iou:.3f}")
ax[2].scatter(xJ, yJ, s=8, c=lab, cmap="viridis", linewidths=0); ax[2].set_title("Visium spd ref")
for a in ax: a.set_aspect("equal"); a.invert_yaxis()
plt.tight_layout(); plt.show()


## 6 - Rasterize the layer signal

`rasterize_with_signal` places a Gaussian kernel at each cell weighted by the layer value,
giving a smooth pia->WM gradient image for source (Xenium) and target (Visium).

In [ ]:
# ============================================================
#  Cell 6 - rasterize layer signals to images I (source), J (target)
# ============================================================
def rasterize_signal(x, y, s, dx, blur=2.0):
    X, Y, img = STalign.rasterize_with_signal(x, y, s=np.asarray(s,float), dx=dx, blur=blur)
    return X, Y, STalign.normalize(np.repeat(img, 3, axis=0))   # 3 identical channels
XI, YI, I = rasterize_signal(xI, yI, layer_axis, DX_SOURCE)
XJ, YJ, J = rasterize_signal(xJ, yJ, lab,        DX_TARGET)
print(f"I {I.shape}  J {J.shape}")
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].imshow(I[0], cmap="magma", origin="lower"); ax[0].set_title("source I (Xenium layer axis)")
ax[1].imshow(J[0], cmap="magma", origin="lower"); ax[1].set_title("target J (Visium spd)")
for a in ax: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()


## 7 - LDDMM (both mirrors, matched fraction decides)

Runs LDDMM from each mirror. The winning mirror has the higher matched fraction (WM > 0.85);
for Br8667 this is **mirror = False** (~74% vs ~4%). `sigmaR` is lowered from the default
5e5 to 1e5 and `a` from 500 to 250 to permit the larger, finer deformation needed for the
GM/WM proportion mismatch between the two sections.

In [ ]:
# ============================================================
#  Cell 7 - LDDMM from both mirrors; keep the higher matched fraction
# ============================================================
LDDMM_KW = dict(niter=2000, diffeo_start=300, a=250.0,
                sigmaM=0.3, sigmaB=0.3, sigmaA=0.3, sigmaR=1e5,
                epL=1e-8, epT=2e-1, epV=3e1,
                muB=torch.zeros(3, device=device), device=device)
runs = {}
for m in (False, True):
    print(f"\n===== mirror={m} =====")
    A_try = torch.tensor(cands[m][0], dtype=DTYPE, device=device)
    out = STalign.LDDMM([YI,XI], I, [YJ,XJ], J, A=A_try, **LDDMM_KW)
    tp = STalign.transform_points_source_to_target(out["xv"], out["v"], out["A"],
                                                   np.stack([yI,xI],1))
    tp = (tp.cpu() if tp.is_cuda else tp).numpy()
    w = STalign.interp([YJ,XJ], out["WM"][None].float(),
                       torch.tensor(tp)[None].permute(-1,0,1).float())
    w = (w.cpu() if w.is_cuda else w)[0,0].numpy()
    runs[m] = dict(out=out, tp=tp, wm=w)
    print(f"matched (WM>{MATCH_THRESH}): {(w>MATCH_THRESH).mean()*100:.1f}%")
best = max(runs, key=lambda m: (runs[m]["wm"] > MATCH_THRESH).mean())
tp, wm = runs[best]["tp"], runs[best]["wm"]
xI_t, yI_t = tp[:,1], tp[:,0]
adata.obs["match_prob"] = wm; adata.obsm["aligned"] = np.c_[xI_t, yI_t]
fr = {m:(runs[m]["wm"]>MATCH_THRESH).mean() for m in runs}
print(f"\n>>> mirror={best} | matched {fr[best]*100:.1f}% (other {fr[not best]*100:.1f}%) <<<")
print(f"[{'PASS' if abs(fr[True]-fr[False])>0.15 else 'CHECK'}] one mirror clearly wins")


## 8 - Non-circular layer validation

For every matched Xenium cell, find the Visium spot it lands in and compare that spot's `spd`
layer with the cell's **own Lieber layer**. Both label sets are independent (Xenium:
label-transfer/BANKSY; Visium: PRECAST), so the confusion matrix is a non-circular measure of
alignment quality. A permutation baseline quantifies the lift.

In [ ]:
# ============================================================
#  Cell 8 - Xenium-Lieber-layer vs landed-Visium-spd confusion
# ============================================================
kd = cKDTree(np.c_[xJ,yJ]); d, jsp = kd.query(np.c_[xI_t,yI_t])
keep = wm > MATCH_THRESH
in_spot = (d <= SPOT_R_PX) & keep & adata.obs["layer"].notna().values
XL2O = {"L1/M":0,"L2/3":1,"L3/4":2,"L5":3,"L6":4,"WM":5,"WMtz":6}
vis_spd = vadata.obs["spd_label"].map({d_:i for i,d_ in enumerate(SPD_ORDER)}).to_numpy()
xo = adata.obs["layer"].map(XL2O).to_numpy()[in_spot].astype(int)   # Xenium's own layer
vo = vis_spd[jsp[in_spot]].astype(int)                              # Visium spot landed on

same = (xo==vo).mean(); adj = (np.abs(xo-vo)<=1).mean(); rho_l = spearmanr(xo,vo).statistic
rng = np.random.default_rng(0)
base_adj = np.array([(np.abs(rng.permutation(xo)-vo)<=1).mean() for _ in range(200)])
lift = (adj - base_adj.mean())/base_adj.std()
print(f"layer agreement: exact {same*100:.0f}% | +/-1 {adj*100:.0f}% | rho {rho_l:.2f}")
print(f"  +/-1 vs random {base_adj.mean()*100:.0f}% -> {lift:.0f} sigma")

C = np.zeros((7,7)); np.add.at(C,(xo,vo),1); Cn = C/C.sum(1,keepdims=True).clip(1)
fig, ax = plt.subplots(1, 2, figsize=(13, 5.5))
im = ax[0].imshow(Cn, cmap="magma", vmin=0, vmax=1)
ax[0].set_xticks(range(7)); ax[0].set_xticklabels(list(XL2O), rotation=45, fontsize=7)
ax[0].set_yticks(range(7)); ax[0].set_yticklabels(list(XL2O), fontsize=7)
ax[0].set_xlabel("landed Visium spd"); ax[0].set_ylabel("Xenium Lieber layer")
ax[0].set_title(f"layer confusion (exact {same*100:.0f}%, +/-1 {adj*100:.0f}%)")
plt.colorbar(im, ax=ax[0], fraction=.046, pad=.02)
p = ax[1].scatter(xI_t, yI_t, s=1, c=wm, cmap="RdYlGn", vmin=0, vmax=1, linewidths=0)
ax[1].set_title(f"match prob ({keep.mean()*100:.0f}% > {MATCH_THRESH})")
ax[1].set_aspect("equal"); ax[1].invert_yaxis(); plt.colorbar(p, ax=ax[1], fraction=.046, pad=.02)
plt.tight_layout(); plt.show()


## 9 - Shared-gene expression concordance (200 um pixels)

Paper-style evaluation: aggregate cells/spots into 200 um pixels, then per gene compute
**cosine and Spearman** between matched pixels, stratified by Moran's I (patterned vs not).

**Report both.** On this cross-platform pair cosine is mean-dominated and unreliable
(e.g. ARPP19 cosine 0.94 but Spearman -0.15); Spearman reflects true spatial correspondence.
Genes with Spearman > 0.3 are essentially all WM/oligodendrocyte markers -- the alignment
recovers the GM/WM axis, not intra-GM layers.

In [ ]:
# ============================================================
#  Cell 9 - 200 um pixel aggregation + cosine/Spearman concordance
# ============================================================
Xk = adata.X[keep]; Xk = Xk.toarray() if hasattr(Xk,"toarray") else np.asarray(Xk)
xa, ya = xI_t[keep], yI_t[keep]
gp = GRID_UM * PX_PER_UM
gid = lambda x,y:(np.floor((x-xJ.min())/gp).astype(int)*100000 +
                  np.floor((y-yJ.min())/gp).astype(int))
gx, gv = gid(xa,ya), gid(xJ,yJ); common = np.intersect1d(np.unique(gx),np.unique(gv))
Vc = vadata.X; Vc = Vc.toarray() if hasattr(Vc,"toarray") else np.asarray(Vc)
xi_g=[adata.var_names.get_loc(g) for g in shared]; vi_g=[vadata.var_names.get_loc(g) for g in shared]
def agg(M,gg,cols):
    o=np.zeros((len(common),len(cols)))
    for k,c in enumerate(common):
        mm=gg==c
        if mm.any(): o[k]=M[np.ix_(mm,cols)].sum(0)
    return o
Pg,Vg = agg(Xk,gx,xi_g), agg(Vc,gv,vi_g)
nx=np.array([(gx==c).sum() for c in common]); ns=np.array([(gv==c).sum() for c in common])
ok=(nx>=10)&(ns>=3); cxy=np.array([[xJ[gv==c].mean(),yJ[gv==c].mean()] for c in common[ok]])
cpm=lambda M:(lambda L:np.log1p(M/np.where(L==0,1,L)*1e4))(M.sum(1,keepdims=True))
Pn,Vn = cpm(Pg[ok]), cpm(Vg[ok])
def mI(v,xy,k=6):
    nb=cKDTree(xy).query(xy,k=k+1)[1][:,1:]; z=v-v.mean(); dd=(z**2).sum()
    return 0. if dd==0 else (z*z[nb].mean(1)).sum()/dd
moran=np.array([mI(Vn[:,j],cxy) for j in range(len(shared))]); pat=moran>0.10
cos=lambda a,b:(a@b/(np.linalg.norm(a)*np.linalg.norm(b))) if a.any() and b.any() else np.nan
csim=np.array([cos(Pn[:,j],Vn[:,j]) for j in range(len(shared))])
ssim=np.array([spearmanr(Pn[:,j],Vn[:,j]).statistic if Pn[:,j].std() and Vn[:,j].std() else np.nan
               for j in range(len(shared))])
res=pd.DataFrame({"gene":shared,"moran":moran,"cosine":csim,"spearman":ssim,
                  "patterned":pat}).sort_values("spearman",ascending=False)
print(f"{ok.sum()} bins @ {GRID_UM:.0f} um, {np.median(nx[ok]):.0f} cells/bin")
print(f"patterned  (n={pat.sum()}): cosine {np.nanmedian(csim[pat]):.2f} | spearman {np.nanmedian(ssim[pat]):.2f}")
print(f"non-patt.  (n={(~pat).sum()}): cosine {np.nanmedian(csim[~pat]):.2f} | spearman {np.nanmedian(ssim[~pat]):.2f}")
print("\ntop concordant genes (all WM/oligo):")
print(res.head(10)[["gene","moran","cosine","spearman"]].to_string(index=False))


## 10 - Summary figure (STalign Fig-3 style)

Cosine + Spearman distributions, the non-circular layer confusion with permutation baseline,
top genes, and a WM-marker spatial map. Saved to `OUT` at 600 dpi (PNG + PDF).

In [ ]:
# ============================================================
#  Cell 10 - publication figure
# ============================================================
OKB = {"pat":"#0072B2","non":"#999999","wm":"#009E73"}
fig = plt.figure(figsize=(19,5.5)); gs = fig.add_gridspec(1,4,wspace=.34)
cl=lambda a:a[~np.isnan(a)]
axA=fig.add_subplot(gs[0,0])
grp=[cl(csim[pat]),cl(csim[~pat]),cl(ssim[pat]),cl(ssim[~pat])]; pos=[1,2,3.4,4.4]
pv=axA.violinplot(grp,positions=pos,showmedians=True,widths=.8)
for b,c in zip(pv["bodies"],[OKB["pat"],OKB["non"]]*2): b.set_facecolor(c); b.set_alpha(.6)
axA.set_xticks([1.5,3.9]); axA.set_xticklabels(["cosine","Spearman"]); axA.axhline(0,c="#bbb",lw=.6)
axA.set_ylabel("similarity"); axA.set_title("a  Shared-gene concordance", loc="left", fontweight="bold")
for p,g in zip(pos,grp): axA.text(p,np.median(g)+.03,f"{np.median(g):.2f}",ha="center",fontsize=8)

axB=fig.add_subplot(gs[0,1]); im=axB.imshow(Cn,cmap="magma",vmin=0,vmax=1)
axB.set_xticks(range(7)); axB.set_xticklabels(list(XL2O),rotation=45,fontsize=7)
axB.set_yticks(range(7)); axB.set_yticklabels(list(XL2O),fontsize=7)
axB.set_xlabel("landed Visium spd"); axB.set_ylabel("Xenium Lieber layer")
axB.set_title(f"b  Layer agreement\n+/-1 {adj*100:.0f}% ({lift:.0f} sigma)", loc="left", fontweight="bold")
plt.colorbar(im,ax=axB,fraction=.046,pad=.02)

axC=fig.add_subplot(gs[0,2]); tb=res.head(12).iloc[::-1]
axC.barh(range(len(tb)),tb["spearman"],color=[OKB["wm"] if p else OKB["non"] for p in tb["patterned"]])
axC.set_yticks(range(len(tb))); axC.set_yticklabels(tb["gene"],fontsize=8)
axC.set_xlabel("Spearman"); axC.set_title("c  Top genes (WM/oligo)", loc="left", fontweight="bold")

axD=fig.add_subplot(gs[0,3])
j=shared.index("MBP"); v=Vn[:,j]
p=axD.scatter(cxy[:,0],cxy[:,1],c=v,s=14,cmap="magma",linewidths=0)
axD.set_title(f"d  MBP (Visium, Moran {moran[j]:.2f})", loc="left", fontweight="bold", fontsize=9)
axD.set_aspect("equal"); axD.invert_yaxis(); axD.set_xticks([]); axD.set_yticks([])
fig.suptitle(f"Xenium->Visium alignment ({SAMPLE}): GM/WM axis recovered, intra-GM limited",
             fontsize=13, fontweight="bold", y=1.02)
for ext in ("png","pdf"):
    fig.savefig(f"{OUT}/fig_alignment_{SAMPLE}.{ext}", dpi=600 if ext=="png" else None, bbox_inches="tight")
plt.show(); print(f"saved -> {OUT}/fig_alignment_{SAMPLE}.png/pdf")


## 11 - Persist results

Writes the aligned + annotated Xenium and the Visium AnnData, plus an alignment metadata dict.
`visium_spot` gives the Xenium-cell -> Visium-spot pairing (for imputation), and `aligned`
holds Visium-pixel coordinates.

In [ ]:
# ============================================================
#  Cell 11 - save annotated AnnData objects
# ============================================================
kd = cKDTree(np.c_[xJ,yJ]); d, jsp = kd.query(np.c_[xI_t,yI_t]); in_spot = d <= SPOT_R_PX
adata.obs["visium_spot"] = np.where(in_spot, np.array(vadata.obs_names)[jsp.clip(0)], None)
adata.uns["alignment"] = dict(
    sample=SAMPLE, target_dx=str(dx), mirror=bool(best), rotation_deg=90.0,
    px_per_um=float(PX_PER_UM), matched_frac=float(keep.mean()), match_thresh=MATCH_THRESH,
    layer_exact=float(same), layer_within1=float(adj), layer_spearman=float(rho_l),
    annotation_source="LieberInstitute/spatialDLPFC_SCZ_XENIUM devel",
    annotation_coverage=float(adata.obs["layer"].notna().mean()))
adata.write(f"{OUT}/{SAMPLE}_xenium_aligned_annotated.h5ad")
vadata.write(f"{OUT}/{SAMPLE}_visium.h5ad")
print(f"written -> {OUT}/")
print(f"  cells {adata.n_obs} | matched {keep.mean()*100:.0f}% | "
      f"annotated {adata.obs['layer'].notna().mean()*100:.0f}% | in-spot {in_spot.mean()*100:.0f}%")
print("  obs: layer, cell_type, banksy_clust, spd, match_prob, visium_spot, orig_row, cell_id_xe")
print("  obsm: spatial (um), aligned (Visium px)")
